# Skin Disease Detection v3 — EfficientNetB0
**11 classes | 3-phase training | Grad-CAM | All bugs fixed**

### Trước khi chạy
- `Runtime` → `Change runtime type` → **T4 GPU**

### Changelog v3
- **[FIX #1]** Bỏ `rescale=1./255` — EfficientNetB0 nhận ảnh 0–255
- **[FIX #2]** Freeze BatchNormalization trong phase 2 và 3
- **[FIX #3]** Sửa `base_model` khi load checkpoint — dùng `model.layers` thay vì `model.layers[1]`
- **[MỚI]** 3-phase training: frozen → fine-tune lr=1e-5 → fine-tune lr=1e-4
- **[MỚI]** Copy data về local trước khi train — nhanh hơn 5–10x
- **[MỚI]** Giảm còn 11 class — bỏ cellulitis (F1=0.36), contact_dermatitis (F1=0.48), psoriasis (F1=0.53)
- **[MỚI]** Grad-CAM tích hợp
- **[MỚI]** Confidence threshold 0.60

## Cell 1 — Mount Drive + kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus)
if not gpus:
    print('CẢNH BÁO: Không có GPU! Vào Runtime > Change runtime type > T4 GPU')

## Cell 2 — Cài thư viện + imports

In [ ]:
!pip install albumentations opencv-python-headless -q

import os, json, glob, random, shutil
import numpy as np
import matplotlib.pyplot as plt
import cv2
import albumentations as A
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
print('Imports OK')

## Cell 3 — Cấu hình

In [ ]:
# =============================================================
# SỬA đường dẫn nếu cần
# =============================================================
DRIVE_DATA_DIR = '/content/drive/MyDrive/Skin Disease Detection/skin_data'
MODEL_DIR      = '/content/drive/MyDrive/Skin Disease Detection/skin_model_v3'
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE             = (224, 224)
BATCH_SIZE           = 32
NUM_EPOCHS_FROZEN    = 20   # phase 1
NUM_EPOCHS_FINETUNE  = 15   # phase 2
NUM_EPOCHS_FINETUNE2 = 20   # phase 3
CONFIDENCE_THRESHOLD = 0.60

with open(os.path.join(DRIVE_DATA_DIR, 'class_names.json')) as f:
    CLASS_NAMES = json.load(f)
NUM_CLASSES = len(CLASS_NAMES)

print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Model dir: {MODEL_DIR}')

## Cell 4 — Copy data về local

**Tại sao cần copy?**
Đọc ảnh trực tiếp từ Drive qua mạng mất ~12s/step → 1 epoch ~50 phút.
Copy về SSD local của Colab trước → ~0.4s/step → 1 epoch ~2 phút.
Copy mất ~8–10 phút nhưng tiết kiệm hàng giờ train.

In [ ]:
LOCAL_DIR = '/content/skin_data'

if not os.path.exists(LOCAL_DIR):
    print('Đang copy data từ Drive về Colab local...')
    print('(Mất khoảng 8–10 phút, chỉ cần làm 1 lần/session)')
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DIR)
    print('Copy xong!')
else:
    print('Data đã có local rồi, bỏ qua copy.')

DATA_DIR  = LOCAL_DIR
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'val')
TEST_DIR  = os.path.join(DATA_DIR, 'test')
print(f'DATA_DIR = {DATA_DIR}')

# Verify
for split in ['train', 'val', 'test']:
    path = os.path.join(DATA_DIR, split)
    n_classes = len(os.listdir(path))
    print(f'  {split}: {n_classes} classes')

## Cell 5 — Load data + Augmentation

**[FIX #1] Không rescale ảnh về 0–1**
EfficientNetB0 có preprocessing layer tích hợp, nhận ảnh 0–255.
Nếu rescale trước → model nhận input ~0.004 → không học được (bug gây accuracy 7% ở v1).

**Augmentation skin-tone**: HueSaturation + RGBShift giúp model
generalize tốt hơn trên da vàng/da tối.

In [ ]:
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.6),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=0, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.7),
    A.RGBShift(r_shift_limit=15, g_shift_limit=10, b_shift_limit=5, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    A.ToGray(p=0.05),
    A.GaussNoise(var_limit=(5, 20), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
])

class SkinDataset(Sequence):
    def __init__(self, directory, img_size, batch_size, class_names, aug=None, shuffle=True):
        self.img_size    = img_size
        self.batch_size  = batch_size
        self.class_names = class_names
        self.num_classes = len(class_names)
        self.aug         = aug
        self.shuffle     = shuffle
        self.paths, self.labels = [], []
        for idx, cls in enumerate(class_names):
            cls_dir = os.path.join(directory, cls)
            if not os.path.exists(cls_dir):
                print(f'  CẢNH BÁO: không tìm thấy {cls_dir}')
                continue
            for ext in ('*.jpg','*.jpeg','*.png','*.bmp','*.webp'):
                for fp in glob.glob(os.path.join(cls_dir, ext)):
                    self.paths.append(fp)
                    self.labels.append(idx)
        self.indices = list(range(len(self.paths)))
        if shuffle: random.shuffle(self.indices)
        self.samples = len(self.paths)
        self.classes = np.array(self.labels)
        print(f'  {os.path.basename(directory)}: {self.samples} ảnh, {self.num_classes} classes')

    def __len__(self):
        return int(np.ceil(self.samples / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        X = np.zeros((len(batch_idx), *self.img_size, 3), dtype=np.float32)
        Y = np.zeros((len(batch_idx), self.num_classes), dtype=np.float32)
        for i, bi in enumerate(batch_idx):
            img = np.array(Image.open(self.paths[bi]).convert('RGB').resize(self.img_size))
            if self.aug:
                img = self.aug(image=img)['image']
            # GIỮ NGUYÊN range 0-255 — EfficientNetB0 tự normalize
            X[i] = img.astype(np.float32)
            Y[i, self.labels[bi]] = 1.0
        return X, Y

    def on_epoch_end(self):
        if self.shuffle: random.shuffle(self.indices)

random.seed(42)
np.random.seed(42)

print('Đang load data...')
train_gen = SkinDataset(TRAIN_DIR, IMG_SIZE, BATCH_SIZE, CLASS_NAMES, aug=aug_pipeline, shuffle=True)
val_gen   = SkinDataset(VAL_DIR,   IMG_SIZE, BATCH_SIZE, CLASS_NAMES, aug=None, shuffle=False)
test_gen  = SkinDataset(TEST_DIR,  IMG_SIZE, BATCH_SIZE, CLASS_NAMES, aug=None, shuffle=False)

print(f'\nTổng: train={train_gen.samples}, val={val_gen.samples}, test={test_gen.samples}')

# Verify pixel range
bx, _ = train_gen[0]
print(f'Pixel range: min={bx.min():.1f}, max={bx.max():.1f}')
print('Kỳ vọng: min~0, max~255 (KHÔNG phải 0–1)')

## Cell 6 — Class weights

In [ ]:
labels  = train_gen.classes
cw_arr  = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(cw_arr))

print('Class weights:')
for idx, name in enumerate(CLASS_NAMES):
    bar = '█' * int(cw_arr[idx] * 10)
    print(f'  {name:<30s} {cw_arr[idx]:.3f}  {bar}')

## Cell 7 — Build model

Kiến trúc:
```
Input (224×224×3, float32, range 0-255)
  → EfficientNetB0 preprocessing  (tự normalize)
  → EfficientNetB0 backbone       (frozen ở phase 1)
  → GlobalAveragePooling2D        (7×7×1280 → 1280)
  → BatchNormalization
  → Dropout(0.4)
  → Dense(256, relu)
  → Dropout(0.3)
  → Dense(11, softmax)            (11 classes)
```

In [ ]:
def build_model(num_classes, trainable_base=False):
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = trainable_base
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(base.input, out), base

model, base_model = build_model(NUM_CLASSES, trainable_base=False)

total   = len(model.layers)
trained = sum(1 for l in model.layers if l.trainable)
print(f'Tổng layers  : {total}')
print(f'Trainable    : {trained}  (custom head)')
print(f'Frozen       : {total-trained}  (EfficientNetB0 base)')
print(f'Input shape  : {model.input_shape}')
print(f'Output shape : {model.output_shape}  ← phải là (None, {NUM_CLASSES})')

## Cell 8 — Phase 1: Train custom head (base frozen)

Chỉ train 6 layer custom head, base EfficientNetB0 đóng băng.
Learning rate 1e-3 — lớn vì chỉ train head mới.
Kỳ vọng epoch 1: ~20–30%.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb1 = [
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_DIR,'best_phase1.keras'),
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

print('=== Phase 1: Base frozen ===')
history1 = model.fit(
    train_gen, epochs=NUM_EPOCHS_FROZEN,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=cb1, verbose=1
)
print(f'\nPhase 1 best val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

## Cell 9 — Phase 2: Fine-tune lr=1e-5 (BN frozen)

**[FIX #2] Freeze BatchNormalization**
Unfreeze toàn bộ base NHƯNG freeze lại tất cả BN layers.
BN lưu statistics của ImageNet — nếu train lại sẽ mất kiến thức pretrained.

Learning rate 1e-5 — nhỏ để không phá weights ImageNet.

In [ ]:
# Unfreeze toàn bộ
for layer in model.layers:
    layer.trainable = True

# [FIX #2] Freeze lại tất cả BN layers
bn_frozen = 0
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
        bn_frozen += 1

print(f'BN frozen: {bn_frozen}')
print(f'Trainable: {sum(1 for l in model.layers if l.trainable)}')

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_DIR,'best_phase2.keras'),
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
]

print('\n=== Phase 2: Fine-tune lr=1e-5 ===')
history2 = model.fit(
    train_gen, epochs=NUM_EPOCHS_FINETUNE,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=cb2, verbose=1
)
print(f'\nPhase 2 best val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

## Cell 10 — Phase 3: Fine-tune lr=1e-4 (BN frozen)

Tiếp tục fine-tune với learning rate lớn hơn 10 lần.
Phase 2 với lr=1e-5 tiến chậm và bị stuck — phase 3 với lr=1e-4 giúp model
thoát khỏi local minimum và tăng thêm ~8–10%.

BN vẫn tiếp tục frozen.

In [ ]:
# BN vẫn frozen từ phase 2, chỉ cần recompile với lr mới
for layer in model.layers:
    layer.trainable = True
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb3 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_DIR,'best_final.keras'),
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
]

print('=== Phase 3: Fine-tune lr=1e-4 ===')
history3 = model.fit(
    train_gen, epochs=NUM_EPOCHS_FINETUNE2,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=cb3, verbose=1
)
print(f'\nPhase 3 best val_accuracy: {max(history3.history["val_accuracy"]):.4f}')

## Cell 11 — Plot training curves

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']     + history3.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy'] + history3.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']         + history3.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']     + history3.history['val_loss']
epochs_x = range(1, len(acc) + 1)
p2_start = len(history1.history['accuracy']) + 1
p3_start = p2_start + len(history2.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_x, acc,     label='Train', linewidth=2)
ax1.plot(epochs_x, val_acc, label='Val',   linewidth=2)
ax1.axvline(p2_start, color='gray',  linestyle='--', alpha=0.7, label='Phase 2 start')
ax1.axvline(p3_start, color='orange',linestyle='--', alpha=0.7, label='Phase 3 start')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_ylim([0, 1])

ax2.plot(epochs_x, loss,     label='Train', linewidth=2)
ax2.plot(epochs_x, val_loss, label='Val',   linewidth=2)
ax2.axvline(p2_start, color='gray',  linestyle='--', alpha=0.7, label='Phase 2 start')
ax2.axvline(p3_start, color='orange',linestyle='--', alpha=0.7, label='Phase 3 start')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curve.png'), dpi=150)
plt.show()
print('Saved training_curve.png')

## Cell 12 — Evaluate trên test set

In [ ]:
best_model = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'best_final.keras'))

print('Đang predict test set...')
y_pred_prob = best_model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = test_gen.classes[:len(y_pred)]

test_loss, test_acc = best_model.evaluate(test_gen, verbose=0)
print(f'\n{"="*50}')
print(f'Test accuracy : {test_acc:.4f} ({test_acc*100:.1f}%)')
print(f'Test loss     : {test_loss:.4f}')
print(f'{"="*50}\n')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## Cell 13 — Grad-CAM

Visualize vùng da mà model tập trung khi chẩn đoán.
- Đỏ/vàng = model chú ý nhiều
- Xanh/tím = model ít chú ý

Nếu heatmap tập trung vào vùng tổn thương → model học đúng.
Nếu tập trung vào nền ảnh → model học shortcut.

In [ ]:
def compute_gradcam(model, img_array, class_idx=None):
    last_conv = model.get_layer('top_conv')
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[last_conv.output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        if class_idx is None:
            class_idx = tf.argmax(predictions[0])
        class_score = predictions[:, class_idx]
    grads   = tape.gradient(class_score, conv_outputs)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam     = tf.reduce_sum(conv_outputs[0] * weights, axis=-1)
    cam     = tf.nn.relu(cam).numpy()
    cam     = cv2.resize(cam, (224, 224))
    cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

def overlay_gradcam(original_img, cam, alpha=0.4):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = heatmap * alpha + original_img * (1 - alpha)
    return np.uint8(np.clip(overlay, 0, 255))

def predict_and_gradcam(img_path, model, class_names, threshold=0.60):
    original  = np.array(Image.open(img_path).convert('RGB').resize((224, 224)))
    img_input = np.expand_dims(original.astype(np.float32), axis=0)
    preds     = model.predict(img_input, verbose=0)[0]
    top_idx   = np.argmax(preds)
    top_conf  = preds[top_idx]
    top_name  = class_names[top_idx]

    print(f'Dự đoán: {top_name} ({top_conf*100:.1f}%)')
    if top_conf < threshold:
        print(f'⚠️  Confidence thấp ({top_conf*100:.1f}% < {threshold*100:.0f}%) — kết quả không đáng tin')

    print('\nTop-3:')
    for i in np.argsort(preds)[::-1][:3]:
        print(f'  {class_names[i]:<30s}: {preds[i]*100:.1f}%')

    cam         = compute_gradcam(model, img_input, class_idx=top_idx)
    cam_overlay = overlay_gradcam(original, cam)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(original);      axes[0].set_title('Ảnh gốc');          axes[0].axis('off')
    axes[1].imshow(cam_overlay);   axes[1].set_title(f'Grad-CAM\n{top_name} ({top_conf*100:.1f}%)'); axes[1].axis('off')
    im = axes[2].imshow(cam, cmap='jet', vmin=0, vmax=1)
    axes[2].set_title('Heatmap thuần'); axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    plt.suptitle(f'Grad-CAM — {top_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'gradcam_sample.png'), dpi=150)
    plt.show()
    return preds, cam

print('Grad-CAM ready.')
print('Dùng: predict_and_gradcam(\'path/to/image.jpg\', best_model, CLASS_NAMES)')

In [ ]:
# Test Grad-CAM trên 4 ảnh mẫu từ test set
sample_classes = random.sample(CLASS_NAMES, min(4, NUM_CLASSES))
fig, axes = plt.subplots(len(sample_classes), 3, figsize=(14, 5*len(sample_classes)))

for row, cls in enumerate(sample_classes):
    imgs = glob.glob(os.path.join(TEST_DIR, cls, '*.jpg')) + \
           glob.glob(os.path.join(TEST_DIR, cls, '*.png'))
    if not imgs: continue
    img_path  = random.choice(imgs[:10])
    original  = np.array(Image.open(img_path).convert('RGB').resize((224, 224)))
    img_input = np.expand_dims(original.astype(np.float32), axis=0)
    preds     = best_model.predict(img_input, verbose=0)[0]
    pred_idx  = np.argmax(preds)
    pred_name = CLASS_NAMES[pred_idx]
    conf      = preds[pred_idx]
    cam         = compute_gradcam(best_model, img_input, class_idx=pred_idx)
    cam_overlay = overlay_gradcam(original, cam)
    correct = '✓' if pred_name == cls else '✗'
    axes[row,0].imshow(original);     axes[row,0].set_title(f'True: {cls}', fontsize=10);                           axes[row,0].axis('off')
    axes[row,1].imshow(cam_overlay);  axes[row,1].set_title(f'{correct} Pred: {pred_name}\n({conf*100:.1f}%)', fontsize=10); axes[row,1].axis('off')
    axes[row,2].imshow(cam, cmap='jet'); axes[row,2].set_title('Heatmap', fontsize=10); axes[row,2].axis('off')

plt.suptitle('Grad-CAM Grid', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'gradcam_grid.png'), dpi=150)
plt.show()

## Cell 14 — Lưu model

In [ ]:
model_path = os.path.join(MODEL_DIR, 'skin_disease_model_v3.keras')
best_model.save(model_path)
size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f'Model saved: {model_path} ({size_mb:.1f} MB)')

cn_path = os.path.join(MODEL_DIR, 'class_names.json')
with open(cn_path, 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print(f'class_names.json: {cn_path}')

p1_best = max(history1.history['val_accuracy'])
p2_best = max(history2.history['val_accuracy'])
p3_best = max(history3.history['val_accuracy'])
config = {
    'version'             : 'v3',
    'model'               : 'EfficientNetB0',
    'num_classes'         : NUM_CLASSES,
    'class_names'         : CLASS_NAMES,
    'removed_classes'     : ['psoriasis', 'cellulitis', 'contact_dermatitis'],
    'phase1_val_accuracy' : round(p1_best, 4),
    'phase2_val_accuracy' : round(p2_best, 4),
    'phase3_val_accuracy' : round(p3_best, 4),
    'test_accuracy'       : round(float(test_acc), 4),
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'bugs_fixed'          : [
        'No rescale=1/255 - EfficientNetB0 expects 0-255',
        'BatchNormalization frozen in phase 2 and 3',
        'base_model fix when loading checkpoint'
    ]
}
cfg_path = os.path.join(MODEL_DIR, 'training_config.json')
with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f'Config saved: {cfg_path}')

print(f'\n=== KẾT QUẢ TỔNG KẾT ===')
print(f'Phase 1 best val_accuracy : {p1_best*100:.1f}%')
print(f'Phase 2 best val_accuracy : {p2_best*100:.1f}%')
print(f'Phase 3 best val_accuracy : {p3_best*100:.1f}%')
print(f'Test accuracy             : {test_acc*100:.1f}%')
print(f'\nDownload về máy:')
print(f'  {model_path}')
print(f'  {cn_path}')

## Cell 15 — Test 1 ảnh bất kỳ

In [ ]:
# Thay đường dẫn bằng ảnh bạn muốn test
all_test_imgs = glob.glob(os.path.join(TEST_DIR, '*', '*.jpg'))
if all_test_imgs:
    test_img = random.choice(all_test_imgs)
    print(f'Test ảnh: {test_img}')
    predict_and_gradcam(test_img, best_model, CLASS_NAMES, threshold=CONFIDENCE_THRESHOLD)
else:
    print('Không tìm thấy ảnh test.')